In [1]:
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import random
from PIL import ImageFont

from MonsterNameGenerator import MarkovMonsterNameGenerator
from textGenerateUtils import generate_scientific_name, generate_prompt, generate_description, refine_prompt_with_image
from imageGenerateUtils import get_image, add_caption


In [2]:
# model setup

# 画像モデルは ComfyUI コンテナ側 (./models/comfyui/) に配置する。
# 切り替えたい場合は imageGenerateUtils の DIFFUSION_MODEL / COMFYUI_DIFFUSION_MODEL 環境変数で指定する。

monster_name_filepath = 'monsterNames.txt'

title_font = ImageFont.truetype("ipagp.ttf", 27)
paragraph_font = ImageFont.truetype("ipagp.ttf", 15)
caption_font = ImageFont.truetype("ipagp.ttf", 12)


In [3]:
# ComfyUI サーバーの疎通と、必要なモデルが配置されているかを確認する
import json, urllib.request
from imageGenerateUtils import COMFYUI_URL, DIFFUSION_MODEL, TEXT_ENCODER, VAE

def _choices(node, field):
    info = json.load(urllib.request.urlopen(f"{COMFYUI_URL}/object_info/{node}"))
    return info[node]["input"]["required"][field][0]

print("ComfyUI:", COMFYUI_URL)
for node, field, want in [("UNETLoader", "unet_name", DIFFUSION_MODEL),
                          ("CLIPLoader", "clip_name", TEXT_ENCODER),
                          ("VAELoader", "vae_name", VAE)]:
    found = _choices(node, field)
    print(f"  {node:<12} {want:<32} {'OK' if want in found else 'NOT FOUND -> ' + str(found)}")
    assert want in found, f"{want} が ComfyUI 側に見つかりません"


ComfyUI: http://comfyui:8188
  UNETLoader   novaAnimeAM_v40.safetensors      OK
  CLIPLoader   qwen_3_06b_base.safetensors      OK
  VAELoader    qwen_image_vae.safetensors       OK


In [4]:
name_generator = MarkovMonsterNameGenerator(n=2)
name_generator.train_from_file(monster_name_filepath)

In [5]:
fields = [
    "杉林", "古代林", "畑", "草むら", "花畑", "密林", "水没林","ジャングル","峠","山の麓","樹海","竹林","森","霧の森","熱帯雨林","サバンナ","桜並木","果樹園",
    "洞窟", "鍾乳洞","谷底","岩石地帯","鉱山","荒野","岩の中",
    "雪原","凍土","氷河",
    "旧市街地", "化学工場跡地","都市の下水道", "古城","都市部","廃工場","地下鉄廃線","空中都市",
    "大砂漠", "オアシス",
    "海", "深海", "浅瀬", "砂浜", "汽水域", "川底","孤島","海底遺跡","湖","潮溜まり","地下水路","滝","沈没船","サンゴ礁",
    "成層圏","惑星中心部","溶岩地帯",
    "モンスターの体内"
]
spicies = [
    "生物",
    "鳥",
    "虫",
    "植物",
    "花",
    "草",
    "木",
    "キノコ",
    "魚",
    "爬虫類",
    "哺乳類",
    "両生類",
    "巨大生物",
    "小型生物",
    "草食動物",
    "肉食動物",
    "寄生生物",
    "絶滅危惧種",
    "甲殻類",
    "貝",
    "群生生物",
    "原始生物",
    "人工生命",
    "分類不明の生物"
]

In [ ]:
finalImages = []
for j in tqdm(range(25)):
    name = name_generator.generate()
    field = random.choice(fields)
    if field == "モンスターの体内":
        field = name_generator.generate()+"の体内"
    spicy = random.choice(spicies)
    if spicy in ("貝","草","鳥","魚") and random.randint(0,1)==1:
        name = name + spicy
    target = "{0}にて観測される架空の{1}「{2}」".format(field, spicy, name)
    print(target)
    description = generate_description(target)
    print(description)
    scientific_name = generate_scientific_name(target, description)
    print(scientific_name)
    prompt = generate_prompt(target, description)
    background_draft = get_image(prompt.strip())
    refined_prompt = refine_prompt_with_image(prompt.strip(), background_draft, target, description)
    background = get_image(refined_prompt.strip())

    finalImage = add_caption(name, description, scientific_name, background, title_font, paragraph_font, caption_font)

    finalImage.save("endemic/{0}-{1}.png".format(j,name))
    finalImages.append(finalImage)

  0%|          | 0/25 [00:00<?, ?it/s]

竹林にて観測される架空の群生生物「ボムートク」
竹林の根元で複数の個体が集合して生息する群生生物。
竹の髄から栄養を吸い上げ、独特の金属音を立てて周囲と通信を行う。
外界の振動に敏感で、足元を踏むと一斉に姿を隠す。
群れを形成すると強固な殻を生じ、外敵から身を守る。
未だ生態の全貌は解明されていない。
竹林の根元において複数の個体が融合し、単一の巨大な有機体として振る舞う群生生物。
竹の髄を消化する特殊な酵素を持ち、その分解過程で生成されるガスが独特の共鳴音を生む。
外敵の接近を感知した個体群は音波を同期させて硬く固まり、物理的な衝撃を完全に遮断する。
観測例は極めて少なく、生態系のどの段階で位置づけられるか未だ不明である。
竹林の根元において複数の個体が融合し、単一の巨大な有機体として振る舞う群生生物。
竹の髄を消化する特殊な酵素を持ち、その分解過程で生成されるガスが独特の共鳴音を生む。
外敵の接近を感知した個体群は音波を同期させて硬く固まり、物理的な衝撃を完全に遮断する。
観測例は極めて少なく、生態系のどの段階で位置づけられるか未だ不明である。
Medullacorda synchronus
竹林の根元において複数の個体が粘液質な触手で相互接続し、巨大なネットワーク状の有機体として振る舞う群生生物。
細胞レベルでの融合を可能にする特殊なリグニン分解酵素を分泌し、竹の内部を空洞化させることで栄養を吸収する。
代謝過程で発生するメタンガスが竹の中空構造に溜まり、風圧や振動によって特定の周波数を発する。
この共鳴音は可聴域以下の超低周波であり、外敵への威嚇ではなく、個体群間の情報伝達と物理的防御シールドの生成に用いられる。
外部からの衝撃を受けると、音波共振により硬質化し、鉄板をも弾く強度を発揮する。
生態系における役割は不明だが、竹の寿命を縮める一方で、特定の菌類との共生関係が推測される。
音響的な特徴により、人間による発見は極めて困難である。
竹林の根元において複数の個体が粘液質な触手で相互接続し、巨大なネットワーク状の有機体として振る舞う群生生物。
細胞レベルでの融合を可能にする特殊なリグニン分解酵素を分泌し、竹の内部を空洞化させることで栄養を吸収する。
代謝過程で発生するメタンガスが竹の中空構造に溜まり、風圧や振動によって特定の周波数を発する。
この共鳴音は可

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(40,24))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img in tqdm(zip(axes.flatten(), finalImages)):
    ax.imshow(img)
    ax.axis('off')